# Aula 4 — Aprendeu, decorou, ou não aprendeu o suficiente?

**Disciplina 2 · Aprendizado de Máquina · IASEG**

Até aqui a gente já sabe avaliar um modelo: comparar com o modelo de referência, escolher a métrica,
olhar que tipo de erro ele comete. Hoje a pergunta é outra: **o resultado no treino diz alguma coisa
sobre o que o modelo vai fazer com casos novos?**

| | Etapa | |
|---|---|---|
| 1 | Definição do problema | |
| 2 | Preparação dos dados | |
| 3 | Divisão treino / teste | |
| 4 | Pré-processamento | |
| **5** | **Treinamento do modelo** | **← hoje** |
| 6 | Predição | |
| 7 | Avaliação | |

- **Parte 1** — O Titanic: os dados de hoje
- **Parte 2** — Por que colocar as colunas na mesma escala
- **Parte 3** — A árvore de decisão
- **Parte 4** — Aprendeu, decorou, ou ficou simples demais?

## Configuração

`pandas` para os dados, `matplotlib` para os gráficos. O resto é importado na etapa em que for usado.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

---

# Parte 1 — O Titanic

## Etapa 1 — Definição do problema

> **Dá para prever quem sobreviveu ao naufrágio do Titanic (1912) a partir dos dados da passagem?**

A saída é uma **categoria** (sobreviveu / não sobreviveu) → **classificação**.
Cada linha já tem a resposta escrita → **supervisionado**.

## Etapa 2 — Preparação dos dados

Uma linha, um passageiro. A base é pública.

In [3]:
URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv"

# Leiam o CSV da URL acima (Aula 2: 02_Auditoria_e_Regressao_Bank)
df = ___

df = pd.read_csv(URL)

print('Linhas:', df.shape[0], '| Colunas:', df.shape[1])
df.head()

Linhas: 891 | Colunas: 15


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


### As colunas

| Coluna | O que é |
|---|---|
| `survived` | **a resposta**: 1 = sobreviveu, 0 = não sobreviveu |
| `pclass` | classe da passagem: 1ª, 2ª ou 3ª |
| `sex` | sexo |
| `age` | idade, em anos |
| `sibsp` | quantos irmãos ou cônjuge estavam a bordo |
| `parch` | quantos pais ou filhos estavam a bordo |
| `fare` | quanto a passagem custou |
| `embarked` | porto de embarque (S, C, Q) |
| `class`, `who`, `adult_male`, `deck`, `embark_town`, `alive`, `alone` | outras colunas, que vamos olhar já já |

**Antes de rodar:** de cada 100 passageiros, quantos vocês acham que sobreviveram?

In [5]:
# Calcule aqui a quantidade de sobreviventes usando a coluna 'survived'
sobreviveram = df['survived'].sum()

print('Sobreviveram:', sobreviveram, 'de', len(df))
print(f'De cada 100 passageiros, {100 * sobreviveram / len(df):.0f} sobreviveram')

Sobreviveram: 342 de 891
De cada 100 passageiros, 38 sobreviveram


### Problemas nos dados

Os mesmos da auditoria da Aula 2: valores faltando e informação repetida.

In [ ]:
# Quantos valores faltam em cada coluna? (Aula 2: a auditoria)
___

`age` falta para **177** passageiros; `deck` falta para **688**, quase todos.

E algumas colunas repetem outras. `class` é `pclass` escrita por extenso. E `alive`?

In [ ]:
# 'class' diz a mesma coisa que 'pclass'?
print("'class' repete 'pclass'?  ", (df['class'].map({'First': 1, 'Second': 2, 'Third': 3}) == df['pclass']).all())

# 'alive' ("yes"/"no") diz a mesma coisa que 'survived' (1/0)?
print("'alive' repete 'survived'?", ((df['alive'] == 'yes') == (df['survived'] == 1)).all())

**`alive` é a própria resposta, escrita de novo.** Um modelo que recebesse essa coluna acertaria
100% e não teria aprendido nada sobre o naufrágio. Ela fica de fora. *(A Aula 5 volta a esse tipo de coluna.)*

### O que entra no modelo

As seis colunas que descrevem o passageiro e a passagem: `pclass`, `sex`, `age`, `sibsp`, `parch`, `fare`. Completem a variável "COLUNAS" na célula abaixo.

E os passageiros **sem idade saem da base**. É a decisão mais simples; existem outras
(preencher com a média, por exemplo), mas qualquer uma tem que estar escrita no relatório.

In [ ]:
COLUNAS = _____

# Só as colunas escolhidas e a resposta; tira quem não tem idade
dados = df[COLUNAS + ['survived']].dropna().copy()

print('Passageiros que ficaram:', len(dados), 'de', len(df))

`sex` é texto, e a máquina só trabalha com números. Com só dois valores, vira **uma coluna de 0 e 1**
(é o *one-hot encoding* da Aula 3, com duas categorias): **1 = mulher, 0 = homem**.

In [ ]:
dados['sex'] = (dados['sex'] == 'female').astype(int)

# Separem as entradas e a resposta (Aula 1 e Aula 3)
X = ___   # as entradas (features): as colunas de COLUNAS
y = ___   # o alvo a prever (a classe): 'survived'

print('X:', X.shape, '| y:', y.shape)
print(f'De cada 100 passageiros que ficaram, {100 * y.mean():.0f} sobreviveram')
X.head()

## Etapa 3 — Divisão em treino e teste

- `test_size=0.3` → 30% dos passageiros ficam guardados para o teste.
- `random_state=42` → fixa o sorteio (qualquer número serve): todo mundo na sala tem **a mesma divisão**.
- `stratify=y` → mantém a mesma proporção de sobreviventes no treino e no teste.

In [ ]:
from sklearn.model_selection import train_test_split

# Dividam com os três parâmetros descritos acima (Aula 1: 01_Pipeline_ML_Iris; Aula 3)
X_train, X_test, y_train, y_test = train_test_split(___)

print('Treino:', len(X_train), 'passageiros | Teste:', len(X_test), 'passageiros')
print(f'No teste, cada passageiro vale {100 / len(X_test):.2f} pontos de acurácia')

### Antes de qualquer modelo: a referência

O chute mais simples, que não aprende nada com as colunas. Todo modelo de hoje é comparado com ele.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score

# Qual é o chute mais simples para uma categoria? (Aula 3)
# Opções: "most_frequent" (a mais comum), "uniform" (sorteio)
referencia = DummyClassifier(strategy="___")
referencia.fit(X_train, y_train)

acc_ref = accuracy_score(y_test, referencia.predict(X_test))

print('A referência diz sempre:', referencia.predict(X_test)[0], '(não sobreviveu)')
print(f'Acurácia da referência no teste: {acc_ref:.1%}')

---

# Parte 2 — Por que colocar as colunas na mesma escala

## Etapa 4 — Pré-processamento

O kNN responde olhando os **vizinhos mais próximos**, e "próximo" é uma **distância**:
ele soma as diferenças entre dois passageiros, coluna por coluna.

Olhem a faixa de cada coluna no treino:

In [ ]:
X_train.describe().loc[['min', 'max']]

`fare` vai de 0 a 512. `sex` vai de 0 a 1.

**Antes de rodar:** para o kNN, o que pesa mais na distância: uma diferença de sexo, ou uma diferença de 10 na tarifa?

### O kNN sem escala

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

# Criem a variável KNN aqui embaixo, depois chamem a função para fitá-la nos
# dados de treino

acc_knn = accuracy_score(y_test, knn.predict(X_test))
acertos_knn = (knn.predict(X_test) == y_test).sum()

print(f'kNN sem escala: {acertos_knn} de {len(y_test)} acertos ({acc_knn:.1%})')

Uma passageira do teste: mulher, 1ª classe. Quem são os 5 vizinhos dela?

In [ ]:
# A primeira mulher de 1ª classe do teste
passageira = X_test[(X_test['sex'] == 1) & (X_test['pclass'] == 1)].iloc[[0]]
passageira

In [ ]:
# Os 5 passageiros do treino mais próximos dela, do mais perto para o mais longe
vizinhos = knn.kneighbors(passageira, return_distance=False)[0]

X_train.iloc[vizinhos]

**O vizinho mais próximo é um homem.** Coluna por coluna, quanto cada diferença pesa na distância:

In [ ]:
# A diferença entre ela e o vizinho mais próximo, ao quadrado (é assim que a distância soma)
diferenca = (passageira.iloc[0] - X_train.iloc[vizinhos[0]]) ** 2

print(diferenca.round(1))
print('Soma:', round(diferenca.sum()))

Na soma (181), **a tarifa responde por 130, e o sexo por 1.** Para o kNN sem escala,
o sexo quase não existe. E não é só com ela:

In [ ]:
# Para cada passageiro do teste: o vizinho mais próximo tem o sexo oposto?
mais_proximo = knn.kneighbors(X_test, n_neighbors=1, return_distance=False)[:, 0]
sexo_oposto = (X_train['sex'].values[mais_proximo] != X_test['sex'].values).sum()

print(f'{sexo_oposto} de {len(X_test)} passageiros têm como vizinho mais próximo alguém do sexo oposto')

### Colocando as colunas na mesma escala

A ideia: medir cada diferença em **"quanto aquela coluna costuma variar"**. A tarifa costuma variar
uns 50; a idade, uns 15; o sexo, 0,5. Depois disso, toda coluna fala a mesma língua.

O `StandardScaler` faz isso. Ele **aprende** a escala de cada coluna, e aplica.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler().set_output(transform="pandas")   # devolve uma tabela, com os nomes das colunas

# Em que parte dos dados o scaler deve aprender a escala?
# Opções: X (todos os passageiros), X_train (só o treino)
scaler.fit(___)

X_train_s = scaler.transform(X_train)
X_test_s = scaler.transform(X_test)      # a MESMA escala do treino, aplicada no teste

X_train_s.describe().loc[['min', 'max']].round(1)

### O mesmo kNN, com escala

**Antes de rodar:** os vizinhos da passageira mudam?

In [ ]:
# O mesmo kNN de antes (k=5), agora treinado nos dados ESCALADOS
knn_s = ___
___

acc_knn_s = accuracy_score(y_test, knn_s.predict(X_test_s))
acertos_knn_s = (knn_s.predict(X_test_s) == y_test).sum()

# Os vizinhos da mesma passageira, agora com escala (mostrados na unidade original)
vizinhos_s = knn_s.kneighbors(scaler.transform(passageira), return_distance=False)[0]
X_train.iloc[vizinhos_s]

In [ ]:
mais_proximo_s = knn_s.kneighbors(X_test_s, n_neighbors=1, return_distance=False)[:, 0]
sexo_oposto_s = (X_train['sex'].values[mais_proximo_s] != X_test['sex'].values).sum()

print(f'Vizinho mais próximo do sexo oposto: {sexo_oposto} sem escala, {sexo_oposto_s} com escala')
print(f'kNN sem escala: {acertos_knn} de {len(y_test)} acertos ({acc_knn:.1%})')
print(f'kNN com escala: {acertos_knn_s} de {len(y_test)} acertos ({acc_knn_s:.1%})')
print(f'Referência:     {acc_ref:.1%}')

**Com escala, os cinco vizinhos são mulheres de 1ª classe**, e o kNN acerta 27 passageiros a mais.

Quem precisa de escala: modelos que medem **distância** ou **somam colunas com pesos** (kNN, regressão logística).
Quem não precisa: a **árvore**, que pergunta sobre uma coluna de cada vez. A gente confere na Parte 3.

---

# Parte 3 — A árvore de decisão

## Um problema que uma linha não resolve

Quatro nuvens de pontos nos cantos de um quadrado, duas cores. Cada cor ocupa uma **diagonal**.

In [ ]:
# Não precisa entender este código: ele só cria os pontos.
rng = np.random.default_rng(0)
cantos = [(-1, -1, 0), (1, 1, 0), (-1, 1, 1), (1, -1, 1)]   # (x, y, cor)

pontos = []
for cx, cy, cor in cantos:
    xy = rng.normal([cx, cy], 0.5, (250, 2))
    pontos.append(pd.DataFrame({'x1': xy[:, 0], 'x2': xy[:, 1], 'cor': cor}))
nuvens = pd.concat(pontos, ignore_index=True)

Xn = nuvens[['x1', 'x2']]
yn = nuvens['cor']
Xn_train, Xn_test, yn_train, yn_test = train_test_split(
    Xn, yn, test_size=0.3, random_state=42, stratify=yn)

plt.figure(figsize=(5, 5))
plt.scatter(Xn['x1'], Xn['x2'], c=yn, cmap='coolwarm', s=8)
plt.title('Quatro nuvens, duas cores')
plt.show()

**Antes de rodar:** onde vocês passariam **uma linha** para separar as duas cores?

A regressão logística da Aula 3 é uma soma com pesos: no plano, isso é uma linha.

In [ ]:
from sklearn.linear_model import LogisticRegression

logistica = LogisticRegression()
logistica.fit(Xn_train, yn_train)

print(f'Logística: treino {logistica.score(Xn_train, yn_train):.1%} | teste {logistica.score(Xn_test, yn_test):.1%}')
print('Referência: 50,0% (as duas cores têm o mesmo número de pontos)')

Nenhuma linha resolve. A logística acerta metade, o mesmo que chutar sempre a mesma cor.

A **árvore de decisão** faz outra coisa: uma sequência de perguntas de **sim ou não**,
cada uma sobre **uma coluna**. *"x1 é menor que 0? Se sim: x2 é menor que 0?"*

In [ ]:
from sklearn.tree import DecisionTreeClassifier

arvore_nuvens = DecisionTreeClassifier(max_depth=3, random_state=42)
arvore_nuvens.fit(Xn_train, yn_train)

print(f'Árvore (profundidade 3): treino {arvore_nuvens.score(Xn_train, yn_train):.1%} | teste {arvore_nuvens.score(Xn_test, yn_test):.1%}')

As regiões que cada modelo desenha: a linha da logística, e os cortes da árvore.

In [ ]:
# Não precisa entender este código: ele pinta a resposta do modelo em cada ponto do plano.
def desenha_regioes(modelo, titulo, ax):
    g1, g2 = np.meshgrid(np.linspace(-3, 3, 300), np.linspace(-3, 3, 300))
    grade = pd.DataFrame({'x1': g1.ravel(), 'x2': g2.ravel()})
    ax.contourf(g1, g2, modelo.predict(grade).reshape(g1.shape), alpha=0.25, cmap='coolwarm')
    ax.scatter(Xn_test['x1'], Xn_test['x2'], c=yn_test, cmap='coolwarm', s=8)
    ax.set_title(f'{titulo}\nteste: {modelo.score(Xn_test, yn_test):.1%}')

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
desenha_regioes(logistica, 'Regressão logística', axes[0])
desenha_regioes(arvore_nuvens, 'Árvore, profundidade 3', axes[1])
plt.show()

## A árvore no Titanic

O que a árvore **guarda** depois de treinar: **as perguntas** (qual coluna, qual valor) e **a resposta de cada folha**.
O kNN guarda todos os dados; as regressões guardam pesos; a árvore guarda perguntas.

Uma árvore pequena, de profundidade 2 (duas perguntas seguidas):

In [ ]:
from sklearn.tree import plot_tree

# Como a árvore das nuvens, mas com profundidade 2 e nos dados do Titanic
arvore2 = DecisionTreeClassifier(___)
arvore2.fit(___, ___)

plt.figure(figsize=(12, 6))
plot_tree(arvore2, feature_names=COLUNAS, class_names=['não', 'sim'],
          filled=True, impurity=False, proportion=False)
plt.show()

Como ler: em cada caixa, a pergunta; à esquerda (*True*) quando a resposta é sim, à direita (*False*) quando é não.
`sex <= 0.5` quer dizer **homem** (0 = homem, 1 = mulher). `value = [não, sim]` conta os passageiros do treino que chegaram ali.

A mesma árvore, em números: quantos passageiros do treino caíram em cada folha, e quantos sobreviveram.

In [ ]:
folhas = pd.DataFrame({
    'mulher': X_train['sex'],
    'classe': X_train['pclass'],
    'folha': arvore2.apply(X_train),
    'sobreviveu': y_train,
})

resumo = folhas.groupby('folha').agg(
    mulher=('mulher', 'first'),
    classes=('classe', lambda c: sorted(c.unique())),
    passageiros=('sobreviveu', 'size'),
    sobreviveram=('sobreviveu', 'sum'),
)
resumo['de cada 100'] = (100 * resumo['sobreviveram'] / resumo['passageiros']).round(0)
resumo['a árvore responde'] = np.where(resumo['de cada 100'] > 50, 'sobreviveu', 'não sobreviveu')
resumo

**Lendo um caminho em voz alta:** *é mulher? Sim. Está na 1ª ou 2ª classe? Sim.* → 107 de 113 sobreviveram → **"sobreviveu"**.

Uma folha guarda uma **proporção**, ou seja, uma chance. A resposta sai de um **corte em 50%**,
como na regressão logística da Aula 3. Por isso a mulher de 3ª classe (45 em 100) recebe "não sobreviveu".

In [ ]:
print(f'Árvore, profundidade 2: treino {arvore2.score(X_train, y_train):.1%} | teste {arvore2.score(X_test, y_test):.1%}')
print(f'Referência:                               teste {acc_ref:.1%}')

E a escala? A mesma árvore, treinada nas colunas escaladas da Parte 2:

In [ ]:
arvore2_s = DecisionTreeClassifier(max_depth=2, random_state=42)
arvore2_s.fit(___, y_train)   # quais colunas: com ou sem escala?

print(f'Sem escala: teste {arvore2.score(X_test, y_test):.1%}')
print(f'Com escala: teste {arvore2_s.score(X_test_s, y_test):.1%}')

**O mesmo resultado.** A árvore pergunta sobre uma coluna de cada vez, então mudar a unidade não muda
a resposta. E a primeira pergunta dela é `sex`: justamente a coluna que o kNN sem escala ignorava.

### O botão da árvore: a profundidade

`max_depth` é quantas perguntas seguidas a árvore pode fazer. Como o *k* do kNN, **quem escolhe é quem modela**.
*(Outro botão, `min_samples_leaf`, diz quantos passageiros cada folha precisa ter, no mínimo.)*

E se a gente não colocar limite nenhum?

In [ ]:
arvore_livre = DecisionTreeClassifier(random_state=42)   # sem max_depth: cresce até não ter mais o que separar
arvore_livre.fit(X_train, y_train)

print('Profundidade:', arvore_livre.get_depth(), '| Folhas:', arvore_livre.get_n_leaves(), '| Passageiros no treino:', len(X_train))
print(f'Treino {arvore_livre.score(X_train, y_train):.1%} | teste {arvore_livre.score(X_test, y_test):.1%}')

**Quase 100% no treino, 75% no teste.** A árvore aprendeu, ou decorou?

---

# Parte 4 — Aprendeu, decorou, ou ficou simples demais?

## As quatro nuvens, profundidade por profundidade

A mesma árvore, com cada vez mais perguntas permitidas. Treino e teste lado a lado.

In [ ]:
linhas = []
for profundidade in [1, 2, 3, 10, None]:
    modelo = DecisionTreeClassifier(max_depth=profundidade, random_state=42).fit(Xn_train, yn_train)
    linhas.append({
        'profundidade': 'sem limite' if profundidade is None else profundidade,
        'treino': f'{modelo.score(Xn_train, yn_train):.1%}',
        'teste': f'{modelo.score(Xn_test, yn_test):.1%}',
    })

pd.DataFrame(linhas).set_index('profundidade')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, profundidade in zip(axes, [1, 3, None]):
    modelo = DecisionTreeClassifier(max_depth=profundidade, random_state=42).fit(Xn_train, yn_train)
    nome = 'sem limite' if profundidade is None else profundidade
    desenha_regioes(modelo, f'Profundidade {nome}', ax)
plt.show()

- **Profundidade 1:** uma pergunta só. Treino e teste ruins, e parecidos: perto da referência (50%). **Simples demais**: não aprendeu o padrão.
- **Profundidade 3:** aprendeu o padrão das quatro nuvens. Treino e teste bons, e parecidos.
- **Sem limite:** 100% no treino. As ilhas pequenas em volta de pontos soltos: a árvore **decorou** o treino, inclusive o que era acaso, e o teste caiu.

## A curva do Titanic, construída pela turma

Cada um recebe **uma profundidade**. Troquem o número na célula abaixo, rodem, e digam os dois números para o quadro.

In [ ]:
profundidade = ___   # a profundidade que você recebeu: 1, 2, 3, 4, 5, 6, 8, 10, 12, 15, ou None (sem limite)

minha_arvore = DecisionTreeClassifier(max_depth=profundidade, random_state=42)
minha_arvore.fit(X_train, y_train)

print('Profundidade:', profundidade)
print(f'Treino: {minha_arvore.score(X_train, y_train):.1%}')
print(f'Teste:  {minha_arvore.score(X_test, y_test):.1%}')

### A curva inteira

In [ ]:
profundidades = [1, 2, 3, 4, 5, 6, 8, 10, 12, 15, None]

linhas = []
for p in profundidades:
    modelo = DecisionTreeClassifier(max_depth=p, random_state=42).fit(X_train, y_train)
    linhas.append({'profundidade': 'sem limite' if p is None else p,
                   'treino': modelo.score(X_train, y_train),
                   'teste': modelo.score(X_test, y_test)})
curva = pd.DataFrame(linhas).set_index('profundidade')

(curva * 100).round(1)

In [ ]:
posicoes = range(len(curva))

plt.figure(figsize=(10, 5))
plt.plot(posicoes, curva['treino'] * 100, marker='o', label='treino')
plt.plot(posicoes, curva['teste'] * 100, marker='o', label='teste')
plt.axhline(acc_ref * 100, linestyle='--', color='gray', label='referência')
plt.xticks(posicoes, curva.index)
plt.xlabel('profundidade da árvore')
plt.ylabel('acurácia (%)')
plt.legend()
plt.title('Titanic: treino e teste conforme a árvore cresce')
plt.show()

**Lendo as duas pontas:**

- **À direita**, o treino sobe até 99% e o teste cai para 75%: a árvore **decorou** os passageiros do treino.
- **À esquerda**, uma pergunta só (*é mulher?*) já acerta 77%, bem acima da referência. Não é "não aprendeu nada":
  é **simples demais**. Treino e teste ficam perto um do outro, e os dois abaixo do que poderiam.
- **Treino e teste perto um do outro não basta.** Nas quatro nuvens, com profundidade 1, os dois estavam perto, e perto do chute.

### E qual profundidade escolher?

Nesta divisão, o melhor teste foi com profundidade 5. Mas essa divisão foi **um** sorteio (`random_state=42`).
E se o sorteio tivesse sido outro?

In [ ]:
# A mesma curva, com 20 sorteios diferentes da divisão treino/teste
resultados = []
for sorteio in range(20):
    a_train, a_test, b_train, b_test = train_test_split(
        X, y, test_size=0.3, random_state=sorteio, stratify=y)
    for p in [1, 3, 5, 8, None]:
        modelo = DecisionTreeClassifier(max_depth=p, random_state=42).fit(a_train, b_train)
        resultados.append({'profundidade': 'sem limite' if p is None else p,
                           'teste': modelo.score(a_test, b_test) * 100})

(pd.DataFrame(resultados)
   .groupby('profundidade', sort=False)['teste']
   .agg(['mean', 'min', 'max'])
   .rename(columns={'mean': 'teste, média', 'min': 'pior sorteio', 'max': 'melhor sorteio'})
   .round(1))

Na média dos 20 sorteios, as profundidades de 1 a 8 empatam, e só a árvore sem limite fica claramente pior.
**O "melhor ponto" de uma divisão só é, em parte, sorte.**

E tem um problema maior: se eu escolho a profundidade **olhando o teste**, o teste deixou de ser juiz:
ele ajudou a escolher. **Na Aula 5:** como escolher sem pedir ajuda ao juiz (validação cruzada).

---

## Fechamento

**O modelo aprendeu, decorou, ou não aprendeu o suficiente?** Olhe o treino e o teste **juntos**, e contra a referência:

| Treino | Teste | Leitura |
|---|---|---|
| perto da referência | perto da referência | simples demais: não aprendeu o padrão |
| bom | bom, perto do treino | aprendeu |
| muito alto | bem abaixo do treino | decorou |

**Para o projeto (pergunta 7 do banco):** *como vocês sabem que o modelo não decorou, e que aprendeu alguma coisa?*
São duas metades: a distância entre treino e teste, e a distância até a referência.

**Sugestão para o dataset de vocês:** treinem uma árvore, anotem treino e teste ao lado da referência,
e digam em que ponta da curva o modelo de vocês está.